<a href="https://colab.research.google.com/github/Daprosero/Domain_Adaptation/blob/main/Images/Notebooks/Results_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/ZhangJUJU/OfficeCaltechDomainAdaptation
!pip install -q gdown
!gdown --id 1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf -O ImageCLEF-DA.zip
!git clone https://github.com/Daprosero/Domain_Adaptation.git

fatal: destination path 'OfficeCaltechDomainAdaptation' already exists and is not an empty directory.
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf
From (redirected): https://drive.google.com/uc?id=1G0x-arLRPuE-IKDfkxSa-bvPMltvYVsf&confirm=t&uuid=e774d749-5a7f-4db3-99d5-68a563dafdc4
To: /content/ImageCLEF-DA.zip
100% 225M/225M [00:00<00:00, 265MB/s]
fatal: destination path 'Domain_Adaptation' already exists and is not an empty directory.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.serialization
from torchvision import models, datasets, transforms
from torchvision.models import (
    resnet18, resnet34, resnet50,
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights
)
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Subset, random_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
import pandas as pd
import random
import math
import os
import zipfile
from itertools import permutations
from tqdm import tqdm
import timm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

In [3]:
from Domain_Adaptation.Images.Utils.models import (
    FeatureExtractor,
    Classifier,
    DomainDiscriminator,
    DANN_ResNet,
    ADDA_ResNet,
    CDAN_ResNet,
    CREDA_ResNet,
    GradientReversalFunction,
    GradientReversalLayer,
    CREDALoss
)


In [4]:
torch.serialization.add_safe_globals([
    FeatureExtractor,
    Classifier,
    DomainDiscriminator,
    DANN_ResNet,
    ADDA_ResNet,
    CDAN_ResNet,
    CREDA_ResNet,
    GradientReversalFunction,
    GradientReversalLayer
])

In [5]:

zip_path = "ImageCLEF-DA.zip"
extract_dir = "ImageCLEF-DA"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Verificar contenido
os.listdir(extract_dir)

set_seed(42)
# Transforms
transform_digits = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

transform_clef = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Dataset loading
mnist_full = datasets.MNIST(root='.', train=True, download=True, transform=transform_digits)
usps_full = datasets.USPS(root='.', train=True, download=True, transform=transform_digits)
svhn_full = datasets.SVHN(root='.', split='train', download=True, transform=transform_digits)
#/content/sample_data
I_full = ImageFolder("/content/ImageCLEF-DA/image_CLEF/i", transform=transform_clef)
P_full = ImageFolder("/content/ImageCLEF-DA/image_CLEF/p", transform=transform_clef)
C_full = ImageFolder("/content/ImageCLEF-DA/image_CLEF/c", transform=transform_clef)

A_full = ImageFolder("/content/OfficeCaltechDomainAdaptation/images/amazon", transform=transform_clef)
W_full = ImageFolder("/content/OfficeCaltechDomainAdaptation/images/webcam", transform=transform_clef)
D_full = ImageFolder("/content/OfficeCaltechDomainAdaptation/images/dslr", transform=transform_clef)

# Agrupación
set_1 = {"M": mnist_full, "U": usps_full, "S": svhn_full}
set_2 = {"I": I_full, "P": P_full, "C": C_full}
set_3 = {"A": A_full, "W": W_full, "D": D_full}
sets_all = {
    "MNIST-USPS-SVHN": set_1,
    "ImageCLEF": set_2,
    "Office-Caltech": set_3
}

In [6]:
EXPERIMENT_PROFILES = {
    "resnet18_profile": {
        "backbone": "resnet18",
        "baseline_lr": 1e-3,
        "dann_lr_default": 1e-4,
        "dann_lr_special": 1e-3,
        "adda_phase2_lr_default": 1e-4,
        "adda_phase2_lr_special": 1e-5,
        "cdan_lr": 1e-4,
        "creda_lr_default": 1e-4,
        "creda_lr_special": 1e-3,
        "creda_lambda_default": 0.0001,
        "creda_lambda_special": 0.0001,
        "special_domains": ["M", "S", "U"],
        "large_batch_domains": ["M", "S"],
        "large_batch_size": 1024,
        "val_ratio_baseline": 0.20,
        "val_ratio_da": 0.15,
        "test_ratio_da": 0.15,
        "eval_batch_size": 8,
        "val_batch_size": 256,
        "delta": 20,
        "alpha": 20,
    },

    "resnet50_profile": {
        "backbone": "resnet50",
        "baseline_lr": 1e-4,
        "dann_lr_default": 1e-4,
        "dann_lr_special": 1e-4,
        "adda_phase2_lr_default": 1e-5,
        "adda_phase2_lr_special": 1e-5,
        "cdan_lr": 1e-4,
        "creda_lr_default": 1e-4,
        "creda_lr_special": 1e-4,
        "creda_lambda_default": 0.001,
        "creda_lambda_special": 0.0001,
        "special_domains": ["M", "S", "U"],
        "large_batch_domains": ["M", "S"],
        "large_batch_size": 1024,
        "val_ratio_baseline": 0.20,
        "val_ratio_da": 0.15,
        "test_ratio_da": 0.15,
        "eval_batch_size": 8,
        "val_batch_size": 256,
        "delta": 20,
        "alpha": 20,
    },

    "vit_profile": {
        "backbone": "vit_tiny_patch16_224",
        "baseline_lr": 1e-4,
        "dann_lr_default": 1e-4,
        "dann_lr_special": 1e-4,
        "adda_phase2_lr_default": 1e-5,
        "adda_phase2_lr_special": 1e-5,
        "cdan_lr": 1e-4,
        "creda_lr_default": 1e-4,
        "creda_lr_special": 1e-4,
        "creda_lambda_default": 0.0001,
        "creda_lambda_special": 0.00001,
        "special_domains": ["M", "S", "U"],
        "large_batch_domains": ["M", "S"],
        "large_batch_size": 1024,
        "val_ratio_baseline": 0.20,
        "val_ratio_da": 0.15,
        "test_ratio_da": 0.15,
        "eval_batch_size": 8,
        "val_batch_size": 256,
        "delta": 20,
        "alpha": 20,
    }
}

In [ ]:
import os
from Domain_Adaptation.Images.Utils.training_pipeline import run_all_models
combinations_dict = {
    "MNIST-USPS-SVHN": ("M",'S'),##Admite un par especifico. ("M",'S') o todas las combinaciones (None,None)
    "ImageCLEF": ("I",'C'),
    "Office-Caltech": ("A",'D'),
}

profile_names = [
    "resnet18_profile",
    "resnet50_profile",
    "vit_profile"
]

base_results_dir = "/content/Results"
base_models_dir = "/content/Models"

all_experiment_results = []

for profile_name in profile_names:
    cfg = EXPERIMENT_PROFILES[profile_name]
    backbone_name = cfg["backbone"]

    os.makedirs(f"{base_results_dir}/{backbone_name}", exist_ok=True)
    os.makedirs(f"{base_models_dir}/{backbone_name}", exist_ok=True)

    print(f"\n========== Ejecutando perfil: {profile_name} | backbone: {backbone_name} ==========")

    results_df = run_all_models(
        combinations_dict=combinations_dict,
        sets_all=sets_all,
        cfg=cfg,
        output_dir=f"{base_models_dir}/{backbone_name}/",
        epochs=1
    )

    results_df["ExperimentProfile"] = profile_name
    results_df["Backbone"] = backbone_name

    results_df.to_csv(
        f"{base_results_dir}/{backbone_name}/all_datasets.csv",
        index=False
    )

    all_experiment_results.append(results_df)

final_results_df = pd.concat(all_experiment_results, ignore_index=True)

final_results_df.to_csv(
    f"{base_results_dir}/all_experiments_all_datasets.csv",
    index=False
)

In [8]:
results_df

,Dataset,Source,Target,Test Accuracy,std,ExperimentProfile,Backbone
0,ImageCLEF_Baseline,I,C,59.500000,33.125477,resnet18_profile,resnet18
1,ImageCLEF_DANN,I,C,63.333333,20.031935,resnet18_profile,resnet18
2,ImageCLEF_ADDA,I,C,60.000000,25.561869,resnet18_profile,resnet18
3,ImageCLEF_CDAN,I,C,65.555556,16.713947,resnet18_profile,resnet18
4,ImageCLEF_CREDA,I,C,65.555556,16.713947,resnet18_profile,resnet18
